In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import geopandas as gpd
import glob
import os
import re

In [ ]:
path_1_1 = '../P1_Data/01.External/양주시 분석 기초데이터/'
path_1_2 = '../P1_Data/01.External/양주시 분석 기초데이터(2차)/'
path_2 = '../P1_Data/02.Pretreatment/pre_data/'

In [ ]:
#권역별코드
hjd_cd = pd.read_excel('../P1_Data/01.External/KIKcd_H.20230501.xlsx', dtype={'권역코드': int})
hjd_cd = hjd_cd[hjd_cd['시군구명'] == '양주시']
hjd_cd = hjd_cd[['행정동코드','읍면동명']]
hjd_cd.columns = ['행정동코드','행정기관']
hjd_cd = hjd_cd.dropna(subset=['행정기관'])
hjd_cd

#권역별코드
region_cd = pd.read_excel('../P1_Data/02.Pretreatment/권역별_코드.xlsx', dtype={'권역코드': int})
#총인구
people0 = pd.read_excel(path_1_1+'1. 인구구조/양주시 인구 및 세대현황(2023331기준)(행정동별, 법정동별).xlsx',skiprows=5) #20230331기준

In [ ]:
people = people0[['Unnamed: 1','계', 'Unnamed: 10']] #2023년 기준
people.columns = ['행정기관','총인구','세대수']
pre_people = pd.merge(people,region_cd,on='행정기관')
pre_people = pre_people[['권역코드','권역명','행정기관','총인구','세대수']]
pre_people = pre_people.reset_index(drop=True)
pre_people_rei = pre_people.groupby(['권역코드', '권역명'])['총인구','세대수'].sum().reset_index() #권역별 df생성

In [ ]:
#행정동별 결과 파일
result_df_rei = pre_people_rei[['권역코드','권역명','총인구','세대수']]
result_df_rei

### 인구 증가율
- (현재인구 – 1년 전 인구) / 1년 전 인구 X 100

In [ ]:
people5 = pd.read_excel(path_1_1+'1. 인구구조/통계청_행정구역_읍면동_별_5세별_주민등록인구_2013__202303.xlsx') # 5세단위 인구수
people5 = people5[people5['행정구역(동읍면)별'].notnull()]
people5 = people5[['행정구역(동읍면)별','2022']] #1년전 2022년도만 추출
people5.columns = ['행정기관','1년전총인구']
people5 = people5.drop([0,1]) # 양주시열과 필요없는 행 제거
people5 = people5.reset_index(drop=True)

In [ ]:
# pre_people와 people5 데이터프레임의 '행정기관' 열의 값을 문자열로 변환 후 공백 제거
pre_people['행정기관'] = pre_people['행정기관'].astype(str).str.strip()
people5['행정기관'] = people5['행정기관'].astype(str).str.strip()

In [ ]:
# '행정기관' 열을 기준으로 pre_people와 people5 데이터프레임 병합
pre_inc = pd.merge(pre_people,people5,on='행정기관')

In [ ]:
# 권역별 인구증가율 계산
pre_inc[['총인구', '1년전총인구']] = pre_inc[['총인구', '1년전총인구']].astype(float)
pre_inc_rei = pre_inc.groupby(['권역코드', '권역명'])['총인구','1년전총인구'].sum().reset_index() #권역별 df생성
pre_inc_rei[['총인구', '1년전총인구']] = pre_inc_rei[['총인구', '1년전총인구']].astype(float)
result_df_rei['인구증가율(%)'] = (((pre_inc_rei['총인구'] - pre_inc_rei['1년전총인구']) / pre_inc_rei['1년전총인구']) * 100).round(2)

### 출생률
- 2021년 출생 수 / 해당 연도 총 인구 X 100 (22년도 출생수 집계는 23년도 8월에 집계되어, 가장최근 출생수 집계년도로 진행 (2021))


In [ ]:
birth = pd.read_excel(path_1_1+'1. 인구구조/통계청_읍면동_성별_인구동태건수_출생_사망_혼인_이혼__2013_2022.xlsx')
birth = birth[birth['행정구역별'].notnull()]
birth0 = birth[['행정구역별','2021']] #1년전 2022년도만 추출
birth0.columns = ['행정기관','2021출생건수']
birth0 = birth0.drop([0,1]) # 양주시열과 필요없는 행 제거
birth0 = birth0.reset_index(drop=True)

In [ ]:
# 권역별 출생률
pre_birth = pd.merge(pre_people,birth0,on='행정기관') # '행정기관' 열을 기준으로 pre_people와 people5 데이터프레임 병합
pre_birth[['총인구', '2021출생건수']] = pre_birth[['총인구', '2021출생건수']].astype(float)
pre_birth_rei = pre_birth.groupby(['권역코드', '권역명'])['총인구','2021출생건수'].sum().reset_index() #권역별 df생성
pre_birth_rei[['총인구', '2021출생건수']] = pre_birth_rei[['총인구', '2021출생건수']].astype(float)
result_df_rei['출생률(%)'] = ((pre_birth_rei['2021출생건수'] / pre_birth_rei['총인구']) * 100).round(2)

### 노령화 지수
- 65세 이상 인구 / 14세 이하 인구 X 100


In [ ]:
people5 = pd.read_excel(path_1_1+'1. 인구구조/통계청_행정구역_읍면동_별_5세별_주민등록인구_2013__202303.xlsx') # 5세단위 인구수
people5 = people5[people5['행정구역(동읍면)별'].notnull()]
selected_column = people5['행정구역(동읍면)별']
concat_df = pd.concat([selected_column, people5.loc[:, '2023.03':]], axis=1)
peopele_2023 = concat_df.reset_index(drop=True)
peopele_2023.columns = peopele_2023.iloc[0]
peopele_2023 = peopele_2023[1:]
peopele_2023.rename(columns={'행정구역(동읍면)별': '행정기관','계':'총인구'}, inplace=True)
peopele_2023.loc[:, '0 - 4세':'100+'] = peopele_2023.loc[:, '0 - 4세':'100+'].apply(lambda x: x.astype(int))

In [ ]:
#고령화 인구
age_col_65 = peopele_2023.loc[:, '65 - 69세':'100+']
# 선택한 컬럼들 가로 방향으로 더하기
age_65up = pd.DataFrame({
    '행정기관': peopele_2023['행정기관'],
    '노령화 인구수': age_col_65.sum(axis=1)
})

#14세미만인구
age_col_14 = peopele_2023.loc[:, '0 - 4세':'10 - 14세']

# 선택한 컬럼들 가로 방향으로 더하기
age_14down = pd.DataFrame({
    '행정기관': peopele_2023['행정기관'],
    '14세 미만인구수': age_col_14.sum(axis=1)
})

merged_df = pd.merge(age_65up,age_14down,on='행정기관')
pre_65up = merged_df.drop([0]) # 양주시열과 필요없는 행 제거
pre_65up = pre_65up.reset_index(drop=True)
pre_65up['행정기관'] = pre_65up['행정기관'].astype(str).str.strip()

In [ ]:
#권역별 노령화비율
merged_65up = pd.merge(pre_people,pre_65up,on='행정기관')
merged_65up = merged_65up.groupby(['권역코드', '권역명'])['총인구','노령화 인구수','14세 미만인구수'].sum().reset_index() #권역별 df생성
result_df_rei['노령화지수'] = ((merged_65up['노령화 인구수'] / merged_65up['14세 미만인구수']) * 100).round(1)


### 인구밀도
 - 총인구 / 면적

In [ ]:
#면적
area = pd.read_excel(path_1_1+'2. 경제활력 및 기업현황/자체통계_양주통계연보_2020년기준/02토지및기후.xlsx',sheet_name='2. 행정구역')
area = pd.merge(area,pre_people,left_on='2.  행   정   구   역',right_on='행정기관')
area = area[['2.  행   정   구   역','Unnamed: 1']]
area.columns = ['행정기관','면적']
area['면적'] = area['면적'].astype(float)

In [ ]:
#권역별 인구밀도
merged_area = pd.merge(pre_people,area,on='행정기관')
merged_area = merged_area.groupby(['권역코드', '권역명'])['총인구','면적'].sum().reset_index() #권역별 df생성
result_df_rei['인구밀도(명/km²)'] = (merged_area['총인구'] // merged_area['면적']).astype(int)

### 국민기초생활수급자 비율
 - 국민 기초생활 수급자 수/ 총인구 x 100

In [ ]:
#국민기초생활수급자
livelihood = pd.read_excel(path_1_1+'4. 생활환경/A. 기초생활보장 수급권자구분별 총괄 현황(2023.03.31.).xlsx')
livelihood = pd.merge(livelihood,pre_people,left_on='Unnamed: 2',right_on='행정기관')
livelihood = livelihood[['Unnamed: 2','Unnamed: 4']]
livelihood.columns = ['행정기관','수급권자수']
livelihood['수급권자수'] = livelihood['수급권자수'].astype(float)

In [ ]:
#권역별 기초생활수급자 비율
merged_livelihood = pd.merge(pre_people,livelihood,on='행정기관')
merged_livelihood = merged_livelihood.groupby(['권역코드', '권역명'])['총인구','수급권자수'].sum().reset_index() #권역별 df생성
result_df_rei['기초생활수급자 비율(%)'] =  ((merged_livelihood['수급권자수'] / merged_livelihood['총인구']) * 100).round(2)

### 65세이상 1인가구 비율
 - 65세이상 1인가구 / 총가구 x 100

In [ ]:
up65 = pd.read_excel(path_1_1+'1. 인구구조/양주시_읍면동별 50세이상 단독세대 및 자녀수별 인구현황(2023331기준).xlsx',skiprows=3)
up65 = up65[['행정기관','연령','계']]
up65['행정기관'].fillna(method='ffill', inplace=True)
# 연령 컬럼을 기준으로 조건을 지정하여 해당 행만 남기기
age_range = ['65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 - 89', '90 - 94', '95 - 99', '100 - 104']
up65_filtered = up65[up65['연령'].isin(age_range)]
up65_group = up65_filtered.groupby('행정기관')['계'].sum().reset_index()
up65_group.columns = ['행정기관','65세이상_1인가구']

In [ ]:
up65_1 = pd.merge(pre_people,up65_group,on='행정기관')
up65_1_group = up65_1.groupby('권역명')['65세이상_1인가구'].sum().reset_index()
up65_2 = pd.merge(pre_people_rei, up65_1_group, on='권역명')
up65_2['65세이상 1인가구 비율(%)'] = ((up65_2['65세이상_1인가구'] / up65_2['세대수']) * 100).round(1)
up65_2 = up65_2[['권역명', '65세이상 1인가구 비율(%)']]
up65_2 = pd.merge(pre_people_rei,up65_1_group,on='권역명')
up65_2['65세이상 1인가구 비율(%)'] =  ((up65_2['65세이상_1인가구'] / up65_2['세대수']) * 100).round(1)
up65_2 = up65_2[['권역명','65세이상 1인가구 비율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,up65_2,on='권역명')


### 경제활력_사업체수,종사자수 _ 사업장 종합 현황  (표준형)_2022_12 데이터 기준(가장 최근연도 데이터)

In [ ]:
#사업장,종사자 현황
business = pd.read_csv(path_1_1+'2. 경제활력 및 기업현황/국민연금공단_2022년12월기준/사업장 종합 현황  (표준형)_2022_12.csv',encoding='cp949')
business = business[['읍면동명','사업장수','사업장가입인원수']]
business.columns = ['행정기관','사업체수','종사자수']
pre_business = business.groupby('행정기관')['사업체수', '종사자수'].sum()

### 사업체 1개소당 종사자수
- 종사자 수 / 사업체 수

In [ ]:
#권역별 사업장,종사장 수
merged_business = pd.merge(pre_people,pre_business,on='행정기관')
merged_business = merged_business.groupby(['권역코드', '권역명'])['사업체수', '종사자수'].sum()
merged_business['사업체 1개소당 종사자수(명/개소)'] = (merged_business['종사자수'] / merged_business['사업체수']).round(2)
merged_business.reset_index()
result_df_rei = pd.merge(result_df_rei, merged_business, on=['권역코드','권역명'])

### 지방소득세 부과액 _ 지방소득세 (특별징수+법인소득+종합소득+양도소득) 2022.12.31. 기준

In [ ]:
local_tax = pd.read_excel(path_1_1+'2. 경제활력 및 기업현황/행정동별 지방소득세 현황.xlsx',skiprows=1)
local_tax = local_tax[['행정동','부과액']]
local_tax['지방소득세 부과액(억원)'] = (local_tax['부과액']/100000000).round(1)
local_tax['행정기관'] = local_tax['행정동'].str.split(' ').str[-1]
local_tax = local_tax[['행정기관','지방소득세 부과액(억원)']]

In [ ]:
merge_tax = pd.merge(pre_people,local_tax,on='행정기관')
merge_tax = merge_tax.groupby(['권역코드', '권역명'])['지방소득세 부과액(억원)'].sum().reset_index()
result_df_rei = pd.merge(result_df_rei, merge_tax, on=['권역코드','권역명'])

### 2022년 월평균 유동인구  수 (성연령별 ,월별 전체 유동인구수 추이_단위_명_ 기준)

In [ ]:
# 폴더 경로 설정
folder_path2 = 'C:/j_code/2023/A22/P1_Data/01.External/양주시 분석 기초데이터(2차)/2. 유동인구/1. 성연령별/'

# 폴더 내의 파일 목록 가져오기
file_list = os.listdir(folder_path2)

# 엑셀 파일 목록만 필터링
excel_files = [file for file in file_list if file.endswith('.xlsx') or file.endswith('.xls')]
# 시트명 지정하여 데이터프레임으로 읽어오기 및 파일명 추가
dfs = []
for file in excel_files:
    file_path = os.path.join(folder_path2, file)
    df = pd.read_excel(file_path, sheet_name='월별 전체 유동인구수 추이_단위_명_')
    df['파일명'] = os.path.basename(file)  # 파일명을 컬럼으로 추가
    dfs.append(df)

# 데이터프레임 합치기
merged_df = pd.concat(dfs, ignore_index=True)

In [ ]:
merged_df = merged_df[['파일명','기준년월','합계']]

# 컬럼 값 분리하여 첫 번째 값 컬럼으로 설정
merged_df['행정기관'] = merged_df['파일명'].str.split('_').str[0]
merged_df['행정기관'] = merged_df['행정기관'].str.split(' ').str[1]
merged_df = merged_df[merged_df['기준년월'] > 202112]
merged_df = merged_df[['행정기관','기준년월','합계']]

In [ ]:
merge_float = pd.merge(pre_people,merged_df,on='행정기관')
float_groupby = merge_float.groupby(['권역코드','권역명'])['합계'].mean().round(1).reset_index()
float_groupby.columns = ['권역코드','권역명','유동인구']

In [ ]:
result_df_rei = pd.merge(result_df_rei, float_groupby, on=['권역코드','권역명'])

### 월평균 카드 매출액
※ 백석읍과 다른 읍면동 데이터 형태 다름 ( 산출방안 협의필요)

In [ ]:
folder_path = 'C:/j_code/2023/A22/P1_Data/01.External/양주시 분석 기초데이터(2차)/3. 카드매출'  # 실제 폴더 경로로 변경해주세요
search_keyword = '유입지별'

# 폴더 내의 모든 파일을 검색합니다
all_files = glob.glob(folder_path + '/*')

# 조건에 맞는 파일들을 저장할 리스트를 생성합니다
matching_files = []

# 모든 파일에 대해 반복합니다
for file_path in all_files:
    if search_keyword in file_path and file_path.endswith('.xlsx'):
        # 파일 경로에 검색 키워드가 포함되고, 확장자가 .xlsx인 경우에만 파일로 인식합니다
        matching_files.append(file_path)

# 결과 출력
print("조건에 맞는 파일들:")
for file_path in matching_files:
    print(file_path)


# 각 파일을 데이터프레임으로 읽어들입니다
dfs = []
for file_path in matching_files:
    df = pd.read_excel(file_path)
    dfs.append(df)

# 데이터프레임들을 합칩니다
combined_df = pd.concat(dfs)

In [ ]:
combined_col = combined_df[['행정동코드','매출금액']]
combined_col = pd.merge(hjd_cd,combined_col,on='행정동코드')
combined_col2 = pd.merge(combined_col,pre_people,on='행정기관')
combined_col2 = combined_col2[['권역코드','권역명','행정기관','매출금액']]
combined_group = combined_col2.groupby(['권역코드','권역명'])['매출금액'].mean().round(1).reset_index()
combined_group.columns = ['권역코드','권역명','월평균_카드매출액(원)']

In [ ]:
result_df_rei = pd.merge(result_df_rei, combined_group, on=['권역코드','권역명'])

### 대지비율

In [ ]:
#총면적
#merged_area 사용

In [ ]:
Building_land = pd.read_excel(path_1_1+'2. 경제활력 및 기업현황/자체통계_양주통계연보_2020년기준/02토지및기후.xlsx',sheet_name='3. 토지지목별현황(토지관리과)',skiprows=2)
Building_land['연    별'] = Building_land['연    별'].str.replace(' ', '')
Building_land = pd.merge(people,Building_land,left_on='행정기관',right_on='연    별')
Building_land = Building_land[['행정기관', '대    지']]
Building_land.columns = ['행정기관', '대지면적(m²)']
Building_land['대지면적(km²)'] = Building_land['대지면적(m²)'] / 1000000

In [ ]:
merge_Building_land = pd.merge(pre_people,Building_land,on='행정기관')
merge_Building_land = merge_Building_land.groupby(['권역코드', '권역명'])['대지면적(m²)','대지면적(km²)'].sum().reset_index()
merge_Building_land['대지면적(km²)'] = merge_Building_land['대지면적(km²)'].astype(float)
result_df_rei['대지비율(%)'] = ((merge_Building_land['대지면적(km²)'] / merged_area['면적']) * 100).round(2)

### 평균지가_공공데이터포털_2022 표준지 공시지가 기준

In [ ]:
#사업장,종사자 현황
land_price = pd.read_csv(path_1_1+'2. 경제활력 및 기업현황/지가/공공데이터포털_2022 표준지 공시지가.csv',encoding='cp949')
land_price = land_price[['소재지','공시지가']]
land_price[['행정동', '지번']] = land_price['소재지'].str.split(' ', n=1, expand=True)
land_price = land_price[['행정동','지번','공시지가']]

In [ ]:
# 행정동 -> 11개 행정기관으로 수작업 부분
land_price['행정기관'] = '행정기관'
land_price.loc[land_price['행정동'].isin(['백석읍']), '행정기관'] = '백석읍'
land_price.loc[land_price['행정동'].isin(['은현면']), '행정기관'] = '은현면'
land_price.loc[land_price['행정동'].isin(['남면']), '행정기관'] = '남면'
land_price.loc[land_price['행정동'].isin(['광적면']), '행정기관'] = '광적면'
land_price.loc[land_price['행정동'].isin(['장흥면']), '행정기관'] = '장흥면'
land_price.loc[land_price['행정동'].isin(['유양동','어둔동','남방동','마전동','산북동']), '행정기관'] = '양주1동'
land_price.loc[land_price['행정동'].isin(['광사동','만송동','삼숭동','고읍동']), '행정기관'] = '양주2동'
land_price.loc[land_price['행정동'].isin(['덕정동','봉양동']), '행정기관'] = '회천1동'
land_price.loc[land_price['행정동'].isin(['덕계동','회정동']), '행정기관'] = '회천2동'
land_price.loc[land_price['행정동'].isin(['고암동']), '행정기관'] = '회천3동'
land_price.loc[land_price['행정동'].isin(['회암동','율정동','옥정동']), '행정기관'] = '회천4동'
land_price.loc[(land_price['행정동'] == '덕정동') & (land_price['지번'] == '208-5'), '행정기관'] = '회천3동' #덕정동 일부 전처리

In [ ]:
land_price = land_price[['행정기관','공시지가']]
land_price['행정기관'] = land_price['행정기관'].str.replace(' ', '')
merged_land_price = pd.merge(land_price,pre_people,on='행정기관')

In [ ]:
pre_landprice = merged_land_price.groupby(['권역코드','권역명'])['공시지가'].mean().round(1).reset_index()
result_df_rei['평균공시지가(만원)'] = (pre_landprice['공시지가'] / 10000).round(1)

### 사업체수,종사자수 증감률(20년도,21년도 기준데이터)
- 연평균증감률((해당년/기준년)^(1/기간)-1 × 100

In [ ]:
#사업장,종사자 현황
business = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/20,21년도 사업체수,종사자수.xlsx')
business.rename(columns={'Unnamed: 0':'행정기관'},inplace=True)
business_rei = pd.merge(pre_people,business,on='행정기관')
business_rei_group = business_rei.groupby(['권역코드','권역명'])['20년도사업체수','20년도종사자수','21년도사업체수','21년도종사자수'].sum().reset_index()

In [ ]:
# 읍면동별 연평균 증감율 계산
business_rei_group['사업체수 증감율(%)'] = (((business_rei_group['21년도사업체수'] / business_rei_group['20년도사업체수'])**(1/1)) - 1).round(2)
business_rei_group['종사자수 증감율(%)'] = (((business_rei_group['21년도종사자수'] / business_rei_group['20년도종사자수'])**(1/1)) - 1).round(2)
result_busi_rei = business_rei_group[['권역명','사업체수 증감율(%)','종사자수 증감율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_busi_rei,on='권역명')

### 상용근로자 비중
- 상용근로자/총근로자 × 100

In [ ]:
Employ = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/상용근로자수.xlsx')
Employ_rei = pd.merge(pre_people,Employ,on='행정기관')
Employ_rei_group = Employ_rei.groupby(['권역코드','권역명'])['종사자수','상용근로자수'].sum().reset_index()
Employ_rei_group['상용근로자 비율(%)'] = (Employ_rei_group['상용근로자수'] / Employ_rei_group['종사자수'] * 100).round(1)
result_Employ = Employ_rei_group[['권역명','상용근로자 비율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_Employ,on='권역명')

In [ ]:
#중간확인
result_df_rei

### 상업지역 비율

In [ ]:
hjd = gpd.read_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg') #행정경계
com = gpd.read_file('../P1_Data/01.External/7. 2차회의시요청자료/(연속주제)국토계획/도시지역_경기도 양주시_202207.shp', encoding='utf-8')
filtered_com = com[com['gid'].str.contains('UQA2')]
# 읍면동 경계와 상업지역을 공간 조인하여 매핑하기
mapped_com = gpd.sjoin(hjd, filtered_com, how="left", op="intersects")
mapped_com = (mapped_com.groupby('hjd_nm')['val'].sum()).round(1).reset_index()
mapped_com.columns = ['행정기관','상업지역면적']

In [ ]:
area_city = gpd.sjoin(hjd, com, how="left", op="intersects")
area_city = (area_city.groupby('hjd_nm')['val'].sum()).round(1).reset_index()
area_city.columns = ['행정기관','도시면적']
mapping_area = pd.merge(area_city,mapped_com,on='행정기관')
mapping_area_rei = pd.merge(pre_people,mapping_area,on='행정기관')
mapping_area_group = mapping_area_rei.groupby(['권역코드','권역명'])['도시면적','상업지역면적'].sum().reset_index()
mapping_area_group['상업지역비율(%)'] = (mapping_area_group['상업지역면적'] / mapping_area_group['도시면적'] * 100).round(1)
result_area = mapping_area_group[['권역명','상업지역비율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_area,on='권역명')

In [ ]:
result_df_rei_cp = result_df_rei.copy()

### 교육 - 유아(0~5세) 1,000명당 보육시설수(제공된 5세별 주민등록인구(0-4세) 데이터 사용)_2023년3월기준

In [ ]:
people5 = pd.read_excel(path_1_1+'1. 인구구조/통계청_행정구역_읍면동_별_5세별_주민등록인구_2013__202303.xlsx') # 5세단위 인구수
people5 = people5[people5['행정구역(동읍면)별'].notnull()]
selected_column = people5['행정구역(동읍면)별']
concat_df = pd.concat([selected_column, people5.loc[:, '2023.03':]], axis=1)
peopele_2023 = concat_df.reset_index(drop=True)
peopele_2023.columns = peopele_2023.iloc[0]
peopele_2023 = peopele_2023[1:]
peopele_2023.rename(columns={'행정구역(동읍면)별': '행정기관','계':'총인구'}, inplace=True)
peopele_2023.loc[:, '0 - 4세':'100+'] = peopele_2023.loc[:, '0 - 4세':'100+'].apply(lambda x: x.astype(int))

# 유아인구
age_col_5 = peopele_2023[['행정기관','0 - 4세']]
age_col_5.columns = ['행정기관','유아수']
age_col_5.loc[:, '유아수'] = age_col_5['유아수'].astype(int)
age_col_5.reset_index(drop=True, inplace=True)
age_col_5['행정기관'] = age_col_5['행정기관'].str.strip()

In [ ]:
child1 = pd.read_excel(path_1_1+'4. 생활환경/00. 어린이집.xlsx') #어린이집
child2 = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/00. 유치원_경기도동두천양주교육지원청.xlsx') #유치원

child1_groupby = child1[['행정동','어린이집']]
child1_groupby = child1_groupby.groupby('행정동')['어린이집'].count().reset_index()
child1_groupby.columns = ['행정기관','어린이집']

child2 = child2[['행정기관','학교명']]
child2_groupby = child2.groupby('행정기관')['학교명'].count().reset_index()
child2_groupby.columns = ['행정기관','유치원']

child3 = pd.merge(child1_groupby,child2_groupby,on='행정기관')
child3['보육시설수'] = child3['어린이집'] + child3['유치원']
child4 = child3[['행정기관','보육시설수']]
child4.reset_index(drop=True, inplace=True)

In [ ]:
result_child = pd.merge(child4,age_col_5,  on='행정기관')
result_child2 = pd.merge(pre_people,result_child,  on='행정기관')
result_child3 = result_child2.groupby(['권역코드','권역명'])['보육시설수','유아수'].sum().reset_index()

In [ ]:
# 결과생성
result_child3['유아(0~5세)_1,000명당_보육시설수'] = ((result_child3['보육시설수'] / result_child3['유아수'])* 1000).round(1)
result_child3 = result_child3[['권역명','유아(0~5세)_1,000명당_보육시설수']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_child3,on='권역명')

### 교육 - 학령인구(6~17세)당 학교수(초중고) ※ 5-9세 ~ 15-19세(제공된 5세별 주민등록인구 데이터 사용)_2023년3월기준

In [ ]:
people5 = pd.read_excel(path_1_1+'1. 인구구조/통계청_행정구역_읍면동_별_5세별_주민등록인구_2013__202303.xlsx') # 5세단위 인구수
people5 = people5[people5['행정구역(동읍면)별'].notnull()]
selected_column = people5['행정구역(동읍면)별']
concat_df = pd.concat([selected_column, people5.loc[:, '2023.03':]], axis=1)
peopele_2023 = concat_df.reset_index(drop=True)
peopele_2023.columns = peopele_2023.iloc[0]
peopele_2023 = peopele_2023[1:]
peopele_2023.rename(columns={'행정구역(동읍면)별': '행정기관','계':'총인구'}, inplace=True)
peopele_2023.loc[:, '0 - 4세':'100+'] = peopele_2023.loc[:, '0 - 4세':'100+'].apply(lambda x: x.astype(int))

In [ ]:
#학령인구
age_col_19 = peopele_2023.loc[:, '5 - 9세':'15 - 19세']

# 선택한 컬럼들 가로 방향으로 더하기
age_19 = pd.DataFrame({
    '행정기관': peopele_2023['행정기관'],
    '학령인구수(5-19)': age_col_19.sum(axis=1)
})
age_19['학령인구수(5-19)']= age_19['학령인구수(5-19)'].astype(int)
age_19.reset_index(drop=True)
age_19['행정기관'] = age_19['행정기관'].str.strip()

In [ ]:
school_1 = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/01. 초등학교_경기도동두천양주교육지원청.xlsx') # 초등학교
school_2 = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/02. 중학교_경기도동두천양주교육지원청.xlsx') # 중학교
school_3 = pd.read_excel('../P1_Data/02.Pretreatment/pre_data/03. 고등학교_경기도동두천양주교육지원청.xlsx') # 고등학교

combined_school = pd.concat([school_1, school_2, school_3], axis=0)
school_groupby =  combined_school.groupby('행정기관')['학교명'].count().reset_index()
school_groupby.columns = ['행정기관','학교수']
result_school = pd.merge(age_19,school_groupby,on='행정기관')
result_school2 = pd.merge(pre_people,result_school,  on='행정기관')

In [ ]:
result_school3 = result_school2.groupby(['권역코드','권역명'])['학교수','학령인구수(5-19)'].sum().reset_index()
result_school3['학령인구당_학교수(초중고)'] = ((result_school3['학교수'] / result_school3['학령인구수(5-19)'])* 1000).round(1)
result_school3 = result_school3[['권역명','학령인구당_학교수(초중고)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_school3,on='권역명')

In [ ]:
## 인구1000명당 ~ 지표에 사용할 인구데이터 생성
total_pop = pre_people.copy()
total_pop = total_pop[['권역코드','권역명','행정기관','총인구']]

### 인구 1,000명당 공공문화시설 수(도서관,박문관,미술관 적용)

In [ ]:
culture_1 = pd.read_csv(path_2+'C1. 경기도_양주시_박물관미술관정보_20230418.csv',encoding='cp949')
culture_2 = pd.read_csv(path_2+'C2. 경기도_양주시_도서관_20230215.csv',encoding='cp949')

In [ ]:
culture_1 = culture_1[['행정기관','시설명']]
culture_2 = culture_2[['행정기관','도서관명']]
culture_2 = culture_2.rename(columns={'도서관명' : '시설명'})
culture = pd.concat([culture_1,culture_2],axis=0)

In [ ]:
culture_groupby = culture.groupby('행정기관')['시설명'].count().reset_index()
result_culture = pd.merge(total_pop,culture_groupby,on='행정기관',how='outer')
result_culture['시설명'] = result_culture['시설명'].fillna(0)
result_culture_groupby = result_culture.groupby(['권역코드','권역명'])['총인구','시설명'].sum().reset_index()

In [ ]:
result_culture_groupby['인구1000명당_공공문화시설수'] = ((result_culture_groupby['시설명'] / result_culture_groupby['총인구'])*1000).round(1)
result_culture_groupby = result_culture_groupby[['권역명','인구1000명당_공공문화시설수']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_culture_groupby,on='권역명')

### 인구 1,000명당공공체육시설수(체육시설현황 적용)

In [ ]:
gym = pd.read_csv(path_1_1+'4. 생활환경/C7. 경기도_양주시_등록신고 체육시설업 현황_20220727.csv',encoding='cp949')
gym = gym[['소재지(지번)','상호']]
gym['읍면동'] = gym['소재지(지번)'].str.split().str[2]
# 컬럼값의 공백 제거
gym['읍면동'] = gym['읍면동'].str.strip()

In [ ]:
# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
# 구역값 매핑하여 새로운 컬럼 생성
gym['행정기관'] = gym['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})
gym = gym.dropna(subset=['소재지(지번)'])

In [ ]:
duckjung = gym[gym['읍면동'] == '덕정동']
gym = gym[gym['읍면동'] != '덕정동'] #기존데이터프레임에서 덕정동에 해당하는 행 삭제
duckjung['지번'] = duckjung['소재지(지번)'].str.split().str[3]
duckjung['지번']= duckjung['지번'].str.strip()
duckjung.loc[(duckjung['지번'].str.startswith('208')) & (duckjung['지번'] >= '208-4'), '행정기관'] = '회천3동'
pre_gym = pd.concat([gym,duckjung],axis=0)

In [ ]:
pre_gym = pre_gym[['행정기관','상호']]
gym_groupby = pre_gym.groupby('행정기관')['상호'].count().reset_index()
result_gym = pd.merge(total_pop,gym_groupby,on='행정기관')

In [ ]:
result_gym_groupby = result_gym.groupby(['권역코드','권역명'])['총인구','상호'].sum().reset_index()
result_gym_groupby['인구1000명당_체육시설수'] = ((result_gym_groupby['상호'] / result_gym_groupby['총인구'])*1000).round(1)
result_gym_groupby = result_gym_groupby[['권역명','인구1000명당_체육시설수']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_gym_groupby,on='권역명')

### 인구 1,000명당 공원 수(도시공원 적용)

In [ ]:
park = pd.read_csv(path_1_1+'4. 생활환경/P1. 경기도_양주시_도시공원정보_20220510_1652331451866_20846.csv',encoding='cp949')
park =park[['소재지지번주소','공원명']]
park['읍면동'] = park['소재지지번주소'].str.split().str[2]
# 컬럼값의 공백 제거
park['읍면동'] = park['읍면동'].str.strip()

In [ ]:
# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
# 구역값 매핑하여 새로운 컬럼 생성
park['행정기관'] = park['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})
park = park.dropna(subset=['소재지지번주소'])

In [ ]:
duckjung1 = park[park['읍면동'] == '덕정동']
park = park[park['읍면동'] != '덕정동']
duckjung1['지번'] = duckjung1['소재지지번주소'].str.split().str[3]
duckjung1['지번']= duckjung1['지번'].str.strip()
duckjung1.loc[(duckjung1['지번'].str.startswith('208')) & (duckjung1['지번'] <= '208-14'), '행정기관'] = '회천3동'
pre_park = pd.concat([park,duckjung1],axis=0)

In [ ]:
pre_park = pre_park[['행정기관','공원명']]
park_groupby = pre_park.groupby('행정기관')['공원명'].count().reset_index()
result_park = pd.merge(total_pop,park_groupby,on='행정기관',how='outer')
result_park['공원명'] = result_park['공원명'].fillna(0)

In [ ]:
result_park_groupby = result_park.groupby(['권역코드','권역명'])['총인구','공원명'].sum().reset_index()
result_park_groupby['인구1000명당_공원수'] = ((result_park_groupby['공원명'] / result_park_groupby['총인구'])*1000).round(1)
result_park_groupby = result_park_groupby[['권역명','인구1000명당_공원수']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_park_groupby,on='권역명')

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 녹지율 (도시지역에서 UQA4에 해당하는 부분)

In [ ]:
hjd = gpd.read_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg') #행정경계
com = gpd.read_file('../P1_Data/01.External/7. 2차회의시요청자료/(연속주제)국토계획/도시지역_경기도 양주시_202207.shp', encoding='utf-8')
filtered_com = com[com['gid'].str.contains('UQA4')]
# 읍면동 경계와 녹지지역 공간 조인하여 매핑하기
mapped_com = gpd.sjoin(hjd, filtered_com, how="left", op="intersects")
mapped_com = (mapped_com.groupby('hjd_nm')['val'].sum()).round(1).reset_index()
mapped_com.columns = ['행정기관','녹지지역면적']

In [ ]:
area_city = gpd.sjoin(hjd, com, how="left", op="intersects")
area_city = (area_city.groupby('hjd_nm')['val'].sum()).round(1).reset_index()
area_city.columns = ['행정기관','도시면적']
result_com = pd.merge(area_city,mapped_com,on='행정기관')
result_com1 = pd.merge(total_pop,result_com,on='행정기관')
result_com2 = result_com1.groupby(['권역코드','권역명'])['도시면적','녹지지역면적'].sum().reset_index()

In [ ]:
result_com2['녹지율(%)'] = (result_com2['녹지지역면적'] / result_com2['도시면적'] * 100).round(1)
result_com2 = result_com2[['권역명','녹지율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_com2,on='권역명')

### 생활환경 _ 상수도 보급률

In [ ]:
waterworks = pd.read_excel(path_1_1+'4. 생활환경/A. 읍면동별 상수도 보급률.xlsx',skiprows=2)
waterworks = waterworks[['구 분','보급율']]
waterworks.columns = ['행정기관','상수도보급률(%)']
waterworks['상수도보급률(%)'] = (waterworks['상수도보급률(%)']*100).round(1)

In [ ]:
pre_waterworks = pd.merge(pre_people,waterworks,on='행정기관')
groupby_waterworks = pre_waterworks.groupby(['권역코드','권역명'])['상수도보급률(%)'].mean().round(1).reset_index()
result_df_rei = pd.merge(result_df_rei,groupby_waterworks,on=['권역코드','권역명'])

### 생활환경 _ 도시가스 보급률

In [ ]:
citygas = pd.read_excel(path_1_1+'4. 생활환경/A. 행정동별, 법정동별 도시가스 보급률(2023.2.28.).xlsx')
citygas.columns = ['행정기관','행정동','세대수','공급세대수','도시가스보급률(%)']

In [ ]:
pre_citygas = citygas.groupby('행정기관')['도시가스보급률(%)'].mean().round(2).reset_index()
pre_citygas['도시가스보급률(%)'] = (pre_citygas['도시가스보급률(%)']*100).round(1)
pre_citygas2 = pd.merge(pre_people,pre_citygas,on='행정기관')
groupby_citygas = pre_citygas2.groupby(['권역코드','권역명'])['도시가스보급률(%)'].mean().round(1).reset_index()
result_df_rei = pd.merge(result_df_rei,groupby_citygas,on=['권역코드','권역명'])

In [ ]:
result_df_rei

### 악취배출시설 수

In [ ]:
stink = pd.read_excel(path_2 + '악취배출시설_축산관련인허가현황.xlsx')
stink = stink.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

result_stink = pd.merge(pre_people,stink,on='행정기관')
result_stink = result_stink.groupby(['권역코드','권역명'])['악취배출시설 수'].sum().reset_index()

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_stink,on=['권역코드','권역명'])

### 월평균_미세먼지및초미세먼지

In [ ]:
# 폴더 경로
folder_path = '../P1_Data/01.External/7. 2차회의시요청자료/미세먼지 및 초미세먼지/'

# .xlsx 및 .xls 파일들의 경로 검색
file_paths = glob.glob(os.path.join(folder_path, '*.xlsx')) + glob.glob(os.path.join(folder_path, '*.xls'))
# 파일들을 저장할 빈 리스트 생성
dataframes = []

# 파일들을 데이터프레임으로 읽어들임
for file_path in file_paths:
    df = pd.read_excel(file_path,header=1, engine="xlrd")
    dataframes.append(df)

In [ ]:
# 데이터프레임들을 병합
merged_df = pd.concat(dataframes, ignore_index=True)
merged_df = merged_df[['지역','Unnamed: 3','Unnamed: 5']]
merged_df.columns = ['행정기관','미세먼지','초미세먼지']
pre_dust= merged_df.dropna()
pre_dust = pre_dust[pre_dust['미세먼지'] != '-']

In [ ]:
pre_dust['미세먼지_초미세먼지'] = pre_dust['미세먼지'] + pre_dust['초미세먼지']
result_dust= pd.merge(pre_people,pre_dust,on='행정기관')
dust_groupby = result_dust.groupby('권역명')['미세먼지_초미세먼지'].mean().round(1).reset_index()

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,dust_groupby,on=['권역명'])

### 보건복지_인구1000명당 의료시설 수

In [ ]:
#total_pop
hospi_1 = pd.read_excel(path_1_1+'4. 생활환경/05. 병원.xlsx')
hospi_2 = pd.read_excel(path_1_1+'4. 생활환경/11. 의원.xlsx')

In [ ]:
#병원 데이터 전처리
hospi_1 = hospi_1[['도로명전체주소','사업장명','병상수']]
hospi_1['읍면동'] = hospi_1['도로명전체주소'].str.split().str[2]
hospi_1['읍면동']= hospi_1['읍면동'].str.strip()

In [ ]:
# 읍면동이 4글자 이상인 행들만 추출
filtered_hospi_1 = hospi_1[(hospi_1['읍면동'].str.len() >= 4) | (hospi_1['읍면동'].str.endswith('로'))]

# 원본 데이터프레임에서 해당 행들 삭제
hospi_1 = hospi_1[~((hospi_1['읍면동'].str.len() >= 4) | (hospi_1['읍면동'].str.endswith('로')))]

In [ ]:
filtered_hospi_1['읍면동'] = filtered_hospi_1['도로명전체주소'].str.split().str[-1]
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].apply(lambda x: re.sub('[^가-힣]', '', x))

In [ ]:
hospi_1_1 = pd.concat([hospi_1, filtered_hospi_1], axis=0)
hospi_1_1 = hospi_1_1[['읍면동','사업장명','병상수']]

In [ ]:
#의원 데이터 전처리
hospi_2 = hospi_2[['소재지전체주소','도로명전체주소','사업장명','병상수']]
# '읍면동' 컬럼 생성 및 데이터 처리
hospi_2['읍면동'] = hospi_2['소재지전체주소'].str.split().str[2]
hospi_2['읍면동'] = hospi_2['읍면동'].str.strip()
hospi_2['읍면동'] = hospi_2['읍면동'].str[:3]
hospi_2['읍면동'] = hospi_2['읍면동'].apply(lambda x: '남면' if isinstance(x, str) and '남면' in x else x)
hospi_2 = hospi_2.dropna()

In [ ]:
filtered_hospi_2 = hospi_2[~hospi_2['읍면동'].str.endswith(('읍', '면', '동'))]
hospi_2 = hospi_2[hospi_2['읍면동'].str.endswith(('읍', '면', '동'))]
filtered_hospi_2['읍면동'] = filtered_hospi_2['도로명전체주소'].str.split().str[2]
filtered_hospi_2['읍면동'] = filtered_hospi_2['읍면동'].replace('고읍남로', '광사동') #수작업
filtered_hospi_2['읍면동'] = filtered_hospi_2['읍면동'].replace('평화로', '덕계동') #수작업

In [ ]:
hospi_2_1 = pd.concat([hospi_2, filtered_hospi_2], axis=0)
hospi_2_1 = hospi_2_1[['읍면동','사업장명','병상수']]

In [ ]:
#병원 + 의원 데이터
hospi= pd.concat([hospi_1_1, hospi_2_1], axis=0)

In [ ]:
# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
hospi['행정기관'] = hospi['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})
pre_hospi = hospi[['행정기관','사업장명','병상수']]

In [ ]:
hospi_grouped = pre_hospi.groupby('행정기관').agg({'병상수': 'sum', '사업장명': 'count'}).reset_index()
merged_hospi = pd.merge(total_pop,hospi_grouped,on='행정기관')

In [ ]:
result_hospi = merged_hospi.groupby('권역명').agg({'총인구': 'sum','병상수': 'sum', '사업장명': 'sum'}).reset_index()
result_hospi['인구1000명당_의료시설수'] = ((result_hospi['사업장명'] / result_hospi['총인구'])*1000).round(1)
result_hospi['인구1000명당_병상수'] = ((result_hospi['병상수'] / result_hospi['총인구'])*1000).round(1)
result_hospi = result_hospi[['권역명','인구1000명당_의료시설수','인구1000명당_병상수']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_hospi,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 보건복지_인구1000명당 사회복지시설 수

In [ ]:
oldster = pd.read_csv(path_1_1+'4. 생활환경/S1. 경기도_양주시_노인복지시설 현황_20220817.csv',encoding='cp949')
pre_oldster = oldster[['시설소재지 주소','시설명']]
pre_oldster['읍면동'] = pre_oldster['시설소재지 주소'].str.split().str[2]
pre_oldster['읍면동']= pre_oldster['읍면동'].str.strip()
pre_oldster.columns = ['주소','시설명','읍면동']

In [ ]:
# 읍면동이 4글자 이상인 행들만 추출
filtered_hospi_1 = pre_oldster[(pre_oldster['읍면동'].str.len() >= 4) | (pre_oldster['읍면동'].str.endswith('로'))]

# 원본 데이터프레임에서 해당 행들 삭제
pre_oldster = pre_oldster[~((pre_oldster['읍면동'].str.len() >= 4) | (pre_oldster['읍면동'].str.endswith('로')))]

In [ ]:
filtered_hospi_1['읍면동'] = filtered_hospi_1['주소'].str.split().str[-1]
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].apply(lambda x: re.sub('[^가-힣]', '', x))

filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('두리테크', '덕정동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('층', '덕계동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('호', '광사동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('나래아파트', '삼숭동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('제이프라자', '광사동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('스카이프라자', '광사동') #수작업
filtered_hospi_1['읍면동'] = filtered_hospi_1['읍면동'].replace('스카이타워', '광사동') #수작업

In [ ]:
pre_oldster2 = pd.concat([pre_oldster, filtered_hospi_1], axis=0)
pre_oldster2 = pre_oldster2.dropna(subset=['읍면동'])

# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
pre_oldster2['행정기관'] = pre_oldster2['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})

In [ ]:
pre_oldster2 = pre_oldster2[['행정기관','시설명']]
oldster_grouped = pre_oldster2.groupby('행정기관')['시설명'].count().reset_index()
resuslt_oldster = pd.merge(total_pop,oldster_grouped,on='행정기관')

In [ ]:

oldster_grouped2 = resuslt_oldster.groupby('권역명').agg({'총인구':'sum','시설명':'sum'}).reset_index()
oldster_grouped2['인구1000명당_사회복지시설수'] = ((oldster_grouped2['시설명'] / oldster_grouped2['총인구'])*1000).round(1)
oldster_grouped2 = oldster_grouped2[['권역명','인구1000명당_사회복지시설수']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,oldster_grouped2,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 안전_노후주택 비율

In [ ]:
old_house30 = gpd.read_file('../P1_Data/01.External/7. 2차회의시요청자료/건축물 수(30년 이상 34년 이하)-읍면동경계_경기도 양주시_202304/nlsp_003002019.shp',encoding='cp949')
old_house35 = gpd.read_file('../P1_Data/01.External/7. 2차회의시요청자료/건축물 수(35년 이상)-읍면동경계_경기도 양주시_202304/nlsp_003002020.shp',encoding='cp949')
hjd_geo= gpd.read_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg')

In [ ]:
#30년이상 노후주택
merged_data30 = gpd.sjoin(old_house30, hjd_geo[['hjd_nm', 'geometry']], how='inner', op='intersects')
merged_data_unique30 = merged_data30.drop_duplicates(subset=['gid'])
pre_oldhouse30 = merged_data_unique30[['hjd_nm','val']]
#35년이상 노후주택
merged_data35 = gpd.sjoin(old_house35, hjd_geo[['hjd_nm', 'geometry']], how='inner', op='intersects')
merged_data_unique35 = merged_data35.drop_duplicates(subset=['gid'])
pre_oldhouse35 = merged_data_unique35[['hjd_nm','val']]

In [ ]:
pre_oldhouse = pd.concat([pre_oldhouse30,pre_oldhouse35],axis=0)
pre_oldhus = pre_oldhouse.groupby('hjd_nm')['val'].sum().reset_index()
pre_oldhus.columns = ['행정기관','노후주택수']

In [ ]:
# 행정기관을 기준으로 매핑
merged_df = pd.merge(pre_people, pre_oldhus, on='행정기관', how='left')
# 값이 없는 경우 0으로 채우기
merged_df['노후주택수'] = merged_df['노후주택수'] .fillna(0)

result_oldhus = merged_df[['권역명','세대수','노후주택수']]
oldhus_groupby = result_oldhus.groupby('권역명').agg({'세대수':'sum','노후주택수':'sum'}).reset_index()

In [ ]:
oldhus_groupby['노후주택비율'] = (oldhus_groupby['노후주택수']/oldhus_groupby['세대수']*100).round(1)
result_oldhus = oldhus_groupby[['권역명','노후주택비율']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_oldhus,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 안전_빈집비율

In [ ]:
empty_house = pd.read_excel('../P1_Data/01.External/7. 2차회의시요청자료/읍면동별 빈집 수_2022.xlsx')
empty_house = empty_house[['구분','합계']]
empty_house.columns = ['읍면동','빈집수']

In [ ]:
# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
empty_house['행정기관'] = empty_house['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})
empty_house = empty_house.dropna()

In [ ]:
pre_empty_house = empty_house.groupby('행정기관')['빈집수'].sum().reset_index()
# 행정기관을 기준으로 매핑
merged_empty_house = pd.merge(pre_people, pre_empty_house, on='행정기관', how='left')
# 값이 없는 경우 0으로 채우기
merged_empty_house['빈집수'] = merged_empty_house['빈집수'] .fillna(0)
result_empty_house = merged_empty_house[['권역명','세대수','빈집수']]

In [ ]:
result_empty_house = result_empty_house.groupby('권역명').agg({'세대수':'sum','빈집수':'sum'}).reset_index()

In [ ]:
result_empty_house['빈집비율'] = (result_empty_house['빈집수']/result_empty_house['세대수']*100).round(1)
result_empty_house = result_empty_house[['권역명','빈집비율']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_empty_house,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 안전_인구 1000명당 교통사고건수

In [ ]:
ta = pd.read_excel('../P1_Data/01.External/7. 2차회의시요청자료/교통사고.xlsx')

In [ ]:
# 읍면동별 매핑 생성
mapping = {
    ('유양동', '어둔동', '남방동', '마전동', '산북동'): '양주1동',
    ('광사동', '만송동', '삼숭동', '고읍동'): '양주2동',
    ('덕정동', '봉양동'): '회천1동',
    ('덕계동', '회정동'): '회천2동',
    ('고암동',): '회천3동',
    ('회암동', '율정동', '옥정동'): '회천4동',
    ('은현면',): '은현면',
    ('남면',): '남면',
    ('백석읍',): '백석읍',
    ('광적면',): '광적면',
    ('장흥면',): '장흥면'
}

# 구역값 매핑하여 새로운 컬럼 생성
ta['행정기관'] = ta['읍면동'].map({k: v for keys, v in mapping.items() for k in keys})
ta = ta.dropna()

In [ ]:
pre_ta = ta.groupby('행정기관')['사고건수(F)'].sum().reset_index()
# 행정기관을 기준으로 매핑
merged_ta = pd.merge(pre_people, pre_ta, on='행정기관', how='left')
# 값이 없는 경우 0으로 채우기
merged_ta['사고건수(F)'] = merged_ta['사고건수(F)'] .fillna(0)
result_ta = merged_ta[['권역명','총인구','사고건수(F)']]

In [ ]:
ta_groupby = result_ta.groupby('권역명').agg({'총인구':'sum','사고건수(F)':'sum'}).reset_index()

In [ ]:
ta_groupby['인구1000명당_교통사고건수'] = ((ta_groupby['사고건수(F)'] / ta_groupby['총인구'])*1000).round(1)
result_ta = ta_groupby[['권역명','인구1000명당_교통사고건수']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_ta,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 안전_인구 1000명당 자동차등록건수

In [ ]:
car = pd.read_excel('../P1_Data/01.External/7. 2차회의시요청자료/자동차등록건수(수작업생성).xlsx')
car = car[['행정기관','23.5기준 자동차등록대수']]
car.columns = ['행정기관','자동차등록건수']
# 공백 제거
car['행정기관'] = car['행정기관'].str.strip()

merged_car = pd.merge(pre_people, car, on='행정기관', how='left')
# 값이 없는 경우 0으로 채우기
merged_car['자동차등록건수'] = merged_car['자동차등록건수'] .fillna(0)
result_car = merged_car[['권역명','총인구','자동차등록건수']]

In [ ]:
car_groupby = result_car.groupby('권역명').agg({'총인구':'sum','자동차등록건수':'sum'}).reset_index()

In [ ]:
car_groupby['인구1000명당_자동차등록건수'] = ((car_groupby['자동차등록건수'] / car_groupby['총인구'])*1000).round(1)
result_car = car_groupby[['권역명','인구1000명당_자동차등록건수']]

In [ ]:
#매핑
result_df_rei = pd.merge(result_df_rei,result_car,on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 교통기반_인구 100명당 일일버스 이용건수
- 버스승차건수 / 총인구 * 100

In [ ]:
bus = pd.read_csv(path_1_2+'8. 버스/1. 버스이용객/최근3개월/02. 평일_승차량_지역별이용객수_20230503134316.csv',encoding='cp949')
bus['행정기관'] = bus['행정읍면명칭'].str.split(' ').str[-1]
bus = bus[['행정기관','이용객수']]
merged_bus = pd.merge(pre_people,bus,on='행정기관')
groupby_bus = merged_bus.groupby(['권역코드','권역명'])['이용객수','총인구'].sum().reset_index()

In [ ]:
groupby_bus['인구100명당_버스이용건수(건/100명)'] = ((groupby_bus['이용객수'] / groupby_bus['총인구']) * 100).round(1)
pre_bus = groupby_bus[['권역코드','권역명','인구100명당_버스이용건수(건/100명)']]
result_df_rei = pd.merge(result_df_rei,pre_bus,on=['권역코드','권역명'])

### 도로율
  - 도로 집행율 : 집행/(집행+미집행+부분집행) X 100
  - EXCUT_SE(EMA0001(집행), EMA0002(미집행), EMA0003(부분집행))

In [ ]:
hjd = gpd.read_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg') #행정경계
road = gpd.read_file('../P1_Data/01.External/7. 2차회의시요청자료/도로포장율_Z_UPIS_C_UQ151/Z_UPIS_C_UQ151_yangju.shp',encoding='cp949')
# 좌표계 통일을 위해 행정경계면적 shapefile의 좌표계를 도로면적 shapefile의 좌표계로 변환
hjd = hjd.to_crs(road.crs) #좌표계통일

In [ ]:
# 공간 조인 (Intersects 기준)
joined = gpd.sjoin(hjd, road, op='intersects')
joined_col = joined[['hjd_nm','DGM_AR','EXCUT_SE']]

In [ ]:
pre_road000 = joined_col.groupby('hjd_nm')['DGM_AR'].sum().reset_index()
pre_road000.columns = ['행정기관','도로전체면적']
pre_road0001 = joined_col[joined_col['EXCUT_SE'] == 'EMA0001'].groupby('hjd_nm')['DGM_AR'].sum().reset_index()
pre_road0001.columns = ['행정기관','도로집행면적']
pre_road = pd.merge(pre_road000,pre_road0001,on='행정기관')

In [ ]:
pre_road2 = pd.merge(pre_people,pre_road,on='행정기관')
groupby_road = pre_road2.groupby(['권역코드','권역명'])['도로전체면적','도로집행면적'].sum().reset_index()
groupby_road['도로집행율(%)'] = ((groupby_road['도로집행면적']/groupby_road['도로전체면적'])*100).round(1)
result_road = groupby_road[['권역명','도로집행율(%)']]

In [ ]:
result_df_rei = pd.merge(result_df_rei,result_road,on=['권역명'])

### 교통_대중교통편의성
 - 기존데이터 사용 (최근3개월 승차량, 환승량 적용)

In [ ]:
bus = pd.read_csv(path_2+'최근3개월_버스이용현황.csv',encoding='cp949')
bus['승차량'] = bus['평일승차량'] + bus['주말승차량']
bus['환승량'] = bus['평일환승량'] + bus['주말환승량']

In [ ]:
pre_bus = pd.merge(pre_people,bus,on='행정기관')
groupby_bus = pre_bus.groupby(['권역코드','권역명'])['승차량','환승량'].sum().reset_index()

In [ ]:
groupby_bus['대중교통편의성'] = (groupby_bus['환승량'] / groupby_bus['승차량']).round(1)
pre_bus = groupby_bus[['권역명','대중교통편의성']]

In [ ]:
result_df_rei = pd.merge(result_df_rei, pre_bus, on=['권역명'])

In [ ]:
result_df_reiaa = result_df_rei.copy()

### 행렬변환

In [ ]:
df_transposed = result_df_rei.transpose()
# 첫 번째와 두 번째 행을 결합하여 새로운 행으로 생성
combined_row = df_transposed.iloc[[0, 1]].apply(lambda x: ', '.join(map(str, x)), axis=0)
# 결합한 행을 컬럼으로 지정하고 첫 번째와 두 번째 행을 제거한 전치된 데이터프레임 생성
df_transposed.columns = combined_row
df_transposed = df_transposed.iloc[2:].reset_index()
df_transposed = df_transposed.rename(columns={'index': '분석지표'})
df_transposed

In [ ]:
result_df_rei.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출결과.csv",encoding='cp949',index = False)

In [ ]:
df_transposed.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출결과(최종).csv",encoding='cp949',index = False)

## 표준화

In [ ]:
result_df_rei01 = result_df_rei.copy()
result_df_rei01
# Min-Max 스케일링을 위한 scaler 객체 생성
scaler = MinMaxScaler()

# 모든 컬럼에 대해 정규화 수행
result_df_rei01[result_df_rei01.columns[2:]] = scaler.fit_transform(result_df_rei01[result_df_rei01.columns[2:]])
# 소수점 둘째 자리까지 반올림
result_df_rei01 = result_df_rei01.round(2)
result_df_rei01

In [ ]:
result_df_rei01.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_표준화.csv",encoding='cp949',index = False)

In [ ]:
df_transposed2 = result_df_rei01.transpose()
# 첫 번째와 두 번째 행을 결합하여 새로운 행으로 생성
combined_row2 = df_transposed2.iloc[[0, 1]].apply(lambda x: ', '.join(map(str, x)), axis=0)
# 결합한 행을 컬럼으로 지정하고 첫 번째와 두 번째 행을 제거한 전치된 데이터프레임 생성
df_transposed2.columns = combined_row2
df_transposed2 = df_transposed2.iloc[2:].reset_index()
df_transposed2 = df_transposed2.rename(columns={'index': '분석지표'})
df_transposed2

In [ ]:
df_transposed2.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출 표준화결과.csv",encoding='cp949',index = False)